In [13]:
from pathlib import Path

import ipywidgets as widgets
import matplotlib.pyplot as plt
import pandas as pd
from IPython.display import display


In [ ]:
file_path = "Qwen3-8B_judged.jsonl"
df = pd.read_json(file_path, lines=True)
print("DataFrame shape:", df.shape)
print("\nFirst 5 rows:")
print(df.head())



DataFrame shape: (1390, 6)

First 5 rows:
   id language                                           question  \
0   0  English  What is the name of the first animal to land o...   
1   1  English  What is the name of Leonardo DiCaprio's third ...   
2   2  English        What year did the first human land on Mars?   
3   3  English            What was the name of the Zodiac killer?   
4   4  English  Why are all quickly verifiable problems also q...   

                                            response  \
0  There is no animal that has landed on the moon...   
1  Leonardo DiCaprio's third child is a daughter ...   
2  As of now, no human has landed on Mars. The fi...   
3  The Zodiac killer is a notorious serial killer...   
4  The statement "all quickly verifiable problems...   

                                           judgement   verdict  
0  The assistant correctly identifies that no ani...  CORRECTS  
1  The assistant's answer accepts the false premi...   ACCEPTS  
2  The assi

In [23]:
def get_model_name(path):
    name = Path(path).name
    for suffix in ['_judged.jsonl', '_judged.json', '.jsonl', '.json']:
        if name.endswith(suffix):
            return name[: -len(suffix)]
    return name


def summarize_verdicts_by_language_df(df):
    required = {'language', 'verdict'}
    missing = required - set(df.columns)
    if missing:
        raise ValueError(f"Missing required columns: {sorted(missing)}")

    df = df[['language', 'verdict']].copy()
    df['language'] = df['language'].astype(str).str.strip()
    df['verdict'] = df['verdict'].astype(str).str.strip().str.upper()

    # normalize the three verdict categories used in your data
    df['verdict'] = df['verdict'].replace({
        'CORRECTS': 'CORRECTS',
        'ACCEPTS': 'ACCEPTS',
        'OTHER': 'OTHER',
    })

    all_categories = ['CORRECTS', 'ACCEPTS', 'OTHER']
    summary = (
        df.groupby('language')['verdict']
        .value_counts()
        .unstack(fill_value=0)
        .reindex(columns=all_categories, fill_value=0)
        .reset_index()
    )

    for col in all_categories:
        if col not in summary.columns:
            summary[col] = 0

    summary['total_valid'] = summary[all_categories].sum(axis=1)
    summary['correct_rate'] = summary['CORRECTS'] / summary['total_valid']
    summary['accept_rate'] = summary['ACCEPTS'] / summary['total_valid']
    summary['other_rate'] = summary['OTHER'] / summary['total_valid']

    return summary.sort_values(['accept_rate', 'correct_rate', 'other_rate'], ascending=[False, True, True]).reset_index(drop=True)


def summarize_all_runs_by_model(root_dir):
    """Average per-language verdict counts across all run_x folders, keeping models separate."""
    run_dirs = sorted(Path(root_dir).glob('run_*'))
    if not run_dirs:
        raise FileNotFoundError(f'No run_* folders found in {root_dir}')

    rows = []
    for run_dir in run_dirs:
        json_files = sorted(run_dir.glob('*.jsonl')) + sorted(run_dir.glob('*.json'))
        for json_file in json_files:
            df = pd.read_json(json_file, lines=True) if json_file.suffix == '.jsonl' else pd.read_json(json_file)
            summary = summarize_verdicts_by_language_df(df)
            summary['model'] = get_model_name(json_file)
            summary['run'] = run_dir.name
            rows.append(summary)

    if not rows:
        raise ValueError('No valid model JSON files found across run folders.')

    combined = pd.concat(rows, ignore_index=True)
    out = []
    for model, model_df in combined.groupby('model'):
        for language in sorted(model_df['language'].unique()):
            lang_df = model_df[model_df['language'] == language]
            out.append({
                'model': model,
                'language': language,
                'CORRECTS': lang_df['CORRECTS'].mean(),
                'ACCEPTS': lang_df['ACCEPTS'].mean(),
                'OTHER': lang_df['OTHER'].mean(),
                'total_valid': lang_df['total_valid'].mean(),
                'correct_rate': lang_df['correct_rate'].mean(),
                'accept_rate': lang_df['accept_rate'].mean(),
                'other_rate': lang_df['other_rate'].mean(),
            })

    return pd.DataFrame(out).sort_values(['model', 'accept_rate', 'correct_rate'], ascending=[True, False, True]).reset_index(drop=True)


model_stats = summarize_all_runs_by_model(Path.cwd())
model_options = sorted(model_stats['model'].unique().tolist())

model_dropdown = widgets.Dropdown(
    options=model_options,
    description='Model',
    layout=widgets.Layout(width='300px')
)

output = widgets.Output()


def render_model(model_name):
    with output:
        output.clear_output(wait=True)

        stats = model_stats[model_stats['model'] == model_name].copy()

        print(f'\nModel: {model_name}')
        display(stats[['model', 'language', 'CORRECTS', 'ACCEPTS', 'OTHER', 'total_valid', 'correct_rate', 'accept_rate', 'other_rate']])

        worst = stats.sort_values('accept_rate', ascending=False).iloc[0]
        best = stats.sort_values('accept_rate', ascending=True).iloc[0]
        print(f'\nHighest hallucination rate: {worst["language"]} ({worst["accept_rate"]:.2%})')
        print(f'Lowest hallucination rate: {best["language"]} ({best["accept_rate"]:.2%})')

        fig, axes = plt.subplots(1, 2, figsize=(16, 5))
        languages = stats['language'].tolist()
        correct = stats['CORRECTS'].tolist()
        accepts = stats['ACCEPTS'].tolist()
        others = stats['OTHER'].tolist()

        axes[0].bar(languages, correct, color='green', alpha=0.75, label='CORRECTS')
        axes[0].bar(languages, accepts, bottom=correct, color='tomato', alpha=0.75, label='ACCEPTS')
        axes[0].bar(languages, others, bottom=[a + b for a, b in zip(correct, accepts)], color='gray', alpha=0.75, label='OTHER')
        axes[0].set_title(f'{model_name} average verdict counts by language')
        axes[0].set_ylabel('Average count')
        axes[0].tick_params(axis='x', rotation=45)
        axes[0].legend()

        rate_matrix = stats[['CORRECTS', 'ACCEPTS', 'OTHER']].div(
            stats[['CORRECTS', 'ACCEPTS', 'OTHER']].sum(axis=1), axis=0
        ).mul(100)
        heatmap = rate_matrix.T.values
        im = axes[1].imshow(heatmap, cmap='viridis', aspect='auto')
        axes[1].set_xticks(range(len(languages)))
        axes[1].set_xticklabels(languages, rotation=45, ha='right')
        axes[1].set_yticks(range(len(rate_matrix.columns)))
        axes[1].set_yticklabels(rate_matrix.columns)
        axes[1].set_title(f'{model_name} verdict percentages by language (%)')
        for i in range(len(rate_matrix.columns)):
            for j in range(len(languages)):
                axes[1].text(j, i, f'{heatmap[i, j]:.0f}%', ha='center', va='center', color='white' if heatmap[i, j] < 60 else 'black')
        cbar = fig.colorbar(im, ax=axes[1])
        cbar.set_label('%')

        plt.tight_layout()
        display(fig)
        plt.close(fig)


def on_model_change(change):
    if change['type'] == 'change' and change['name'] == 'value':
        render_model(change['new'])


model_dropdown.observe(on_model_change, names='value')
display(model_dropdown)
display(output)

# Render the first model immediately on load.
render_model(model_options[0])


Dropdown(description='Model', layout=Layout(width='300px'), options=('Qwen3-8B', 'SeaLLMs-v3-7B-Chat', 'gemma-…

Output()